In [1]:
import numpy as np
import pandas as pd
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations
from time import process_time

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)

def makesetflex (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    M = len(B)
    added = []
    for i in range(M):
        new = B[0:i+1]
        for k in range(N):
            if len(A[k])==len(new) and len(np.intersect1d(A[k],new))==len(new):
                break
            if k == N-1:
                A.append(new)
                added.append(new)
    return(A,added)


In [3]:
def robust_counter_powerU (sets,p,R,r,m,r_f,c,rav):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable(M, nonneg= True)
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    w = cp.Variable(N)
    z = cp.Variable(M)
    s = cp.Variable(N)
    f_obj = 0
    constraints = [w - gamma*(np.zeros(N)+1) <= t]
    for i in range(N):
        lbdasum = 0
        vsum = 0
        for j in range(M):
            if i in sets[j]:
                lbdasum = lbdasum + lbda[j]
                vsum = vsum + v[j]
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
        constraints.append(-cp.power(z3,rav)/rav - lbdasum - beta <= 0)
        constraints.append(s[i] == -alpha + vsum)
        constraints.append(cp.kl_div(gamma,w[i])+gamma+s[i]-w[i]<= 0)
    for j in range(M):
        constraints.append(cp.pos(-(1-m)*v[j]+lbda[j]) <= z[j])
    constraints.append(cp.abs(a)<=100)
    constraints.append(alpha + beta + gamma * r  + cp.sum(z) + p@t <= c)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

In [4]:
def robustcheckpoweru(a,R,r,p,m,r_f,rav):
    N = len(p)
    x = -(R.dot(a)+(1-sum(a))*r_f)**rav/rav
    rank = np.argsort(-x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,q_b.value)

In [5]:
def squeeze_algo_putility(R,r,c,p,m,r_f,rav):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=100]
    h = np.zeros(N)
    [iterations, steps, f_obj] = [0,0,0]
    for i in range(N-1):
        h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
    h[N-1]=h_3(p[N-1],m)
    for i in range(N):
        f_obj = cp.power((R @ a)[i]+(1-cp.sum(a))*r_f,rav)/rav*p[i] + f_obj
    constraints.append(-h.T@(cp.power(((R @ a)+ (1-cp.sum(a))*r_f),rav)/rav)<= c)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    [nonstop, nonstop2] = [True, False]
    [lowerobj, gapcut, gapRC] = [-np.inf, np.inf, np.inf]
    [firsttime, firstit] = [True, True]
    while nonstop:
        [rbvalue,h] = robustcheckpoweru(w,R,r,p,m,r_f,rav)
        print('rbvalue',rbvalue)
        if rbvalue <= c:
            if firstit:
                gapcut = iterations
                gapRC = steps
            return(iterations,upperobj, upperobj - lowerobj, gapcut, gapRC)
            #return('cut-stop',w,'upperbound', upperobj, 'lowerbound', lowerobj, 'cuts added', 
                   #iterations,' robust iterations' ,steps, 'cuts before 1E-5 gap', gapcut, 'RCsteps before 1E-5 gap', gapRC)
        if firsttime and rbvalue - c < 0.5:
            oldrank = np.argsort((R.dot(w)+(1-sum(w))*r_f)**rav/rav)
            sets = ranktoset(oldrank)
            nonstop2 = True
            firsttime = False
        constraints.append(-h.T@(cp.power(((R @ a)+ (1-cp.sum(a))*r_f),rav)/rav)<= c)
        iterations = iterations + 1
        while nonstop2:
            [w,lowerobj_new] = robust_counter_powerU (sets,p,R,r,m,r_f,c,rav)
            newrank = np.argsort((R.dot(w)+(1-sum(w))*r_f)**rav/rav)
            if lowerobj_new < lowerobj +1e-5:
                #print('RC_rbvalue',robustcheckpoweru(w,R,r,p,m,r_f,rav)[0])
                break
            lowerobj = lowerobj_new
            oldrank = newrank
            sets = ranktoset(newrank)
            steps = steps + 1
            print('RC steps',steps,'lowerbound',lowerobj)
        nonstop2 = False
        if upperobj - lowerobj <= 1e-5 and firstit:
            gapcut = iterations
            gapRC = steps
            firstit = False
            #return('gap stop', w,'upperbound' , upperobj, 'lowerbound', lowerobj, 'cuts added', iterations,' robust iterations' ,steps)
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value
        if firsttime == False:
            newrank = np.argsort((R.dot(w)+(1-sum(w))*r_f)**rav/rav)
            sets2 = ranktoset(newrank)
            lowerobj_new = robust_counter_powerU (sets2,p,R,r,m,r_f,c,rav)[1]
            if lowerobj_new > lowerobj+1e-5:
                nonstop2 = True
                sets = sets2
                oldrank = newrank
                steps = steps + 1
                print('RC steps from cut',steps)
        #[sets,added] = makesetflex(sets, np.argsort((R.dot(w)+(1-sum(w))*r_f)**rav/rav))
        #print('upperbound' , upperobj, 'lowerbound', lowerobj, 'cuts added', iterations,' robust iterations' ,steps)

In [6]:
def normal_cutting_plane(R,r,c,p,m,r_f,rav):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=100]
    h = np.zeros(N)
    iterations = 0
    f_obj = 0
    for i in range(N-1):
        h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
    h[N-1]=h_3(p[N-1],m)
    for i in range(N):
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
    z1 = ((R @ a)+ (1-cp.sum(a))*r_f)
    constraints.append(-h.T@(cp.power(z1,rav)/rav)<= c)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    nonstop = True
    while nonstop:
        [rbvalue,h] = robustcheckpoweru(w,R,r,p,m,r_f,rav)
        print('rbvalue',rbvalue)
        if rbvalue <= c:
            return('cut-stop',w,'objective', upperobj, 'cuts added', iterations)
        constraints.append(-h.T@(cp.power(z1,rav)/rav)<= c)
        iterations = iterations + 1
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value 
        print('objective',upperobj,'cuts-added', iterations)

In [7]:
def non_robustcheckpoweru(a,R,p,m,r_f,rav):
    N = len(p)
    x = -(R.dot(a)+(1-sum(a))*r_f)**rav/rav
    rank = np.argsort(-x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = p[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value,q_b.value)

def non_robust_cutting_plane(R,c,p,m,r_f,rav):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=100]
    h = np.zeros(N)
    iterations = 0
    f_obj = 0
    for i in range(N-1):
        h[i] = h_3(sum(p[i:N]),m)-h_3(sum(p[i+1:N]),m)
    h[N-1]=h_3(p[N-1],m)
    for i in range(N):
        z3 = (R @ a)[i]+(1-cp.sum(a))*r_f
        f_obj = cp.power(z3,rav)/rav*p[i] + f_obj
    z1 = ((R @ a)+ (1-cp.sum(a))*r_f)
    constraints.append(-h.T@(cp.power(z1,rav)/rav)<= c)
    obj = cp.Maximize(f_obj)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    nonstop = True
    while nonstop:
        [rbvalue,h] = non_robustcheckpoweru(w,R,p,m,r_f,rav)
        #print('rbvalue',rbvalue)
        if rbvalue <= c:
            realrbv = robustcheckpoweru(w,R,r,p,m,r_f,rav)[0]
            #print(realrbv,upperobj)
            return(realrbv,upperobj)
            #return('cut-stop',w,'objective', upperobj, 'cuts added', iterations)
        constraints.append(-h.T@(cp.power(z1,rav)/rav)<= c)
        iterations = iterations + 1
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value 
        #print('objective',upperobj,'cuts-added', iterations)

In [224]:
np.random.seed(5)

In [225]:
N=100
p = np.zeros(N)+1/N
I = 5
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
#print(R)

[0.0945471  0.03368908 0.08936526 0.00499518 0.04724324]


In [226]:
rav= 1-1.1
r = 1
m = 0.95    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.01
c = 17

In [227]:
-r_f**rav/rav

15.848931924611126

In [209]:
squeeze_algo_putility(R,r,c,p,m,r_f,rav)

rbvalue 19.219516518490387
rbvalue 17.98804667618688
rbvalue 17.000000382346556
RC steps 1 lowerbound -15.68332324622295
rbvalue 17.000000357038335
rbvalue 17.000002454174858
rbvalue 16.999999959974453


(5, -1.2645444336101264e-07, 3, 1)

In [212]:
normal_cutting_plane(R,r,c,p,m,r_f,rav)

rbvalue 18.4570025688868
objective -15.626047054908925 cuts-added 1
rbvalue 17.9346002326339
objective -15.63240792712309 cuts-added 2
rbvalue 17.49533581601157
objective -15.634168722172296 cuts-added 3
rbvalue 17.026324709030323
objective -15.634182322567016 cuts-added 4
rbvalue 16.99999974006937


('cut-stop',
 array([ 0.0025965 ,  0.01697496,  0.00600871, -0.00178486,  0.0042767 ,
         0.00382792,  0.00099087,  0.01039416,  0.01003148,  0.00383535]),
 'objective',
 -15.634182322567016,
 'cuts added',
 4)

In [210]:
non_robust_cutting_plane(R,c,p,m,r_f,rav)

rbvalue 17.715034124963317
rbvalue 17.006460728794437
rbvalue 16.99999988116863
18.133908051764525 -15.660337232850704


(18.133908051764525, -15.660337232850704)

In [8]:
T = 20
N=100
p = np.zeros(N)+1/N
I = 5
rav= 1-1.1
r = 1
m = 0.95    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.01
c = 17
[CI, FG, Gap_cut, Gap_RC] = [np.zeros(T), np.zeros(T), np.zeros(T), np.zeros(T)]
[risk_df, obj_df] = [np.zeros(T), np.zeros(T)]
np.random.seed(6)
for i in range(T):
    print(i)
    R = np.random.normal(0.05,0.2,size=(N,I))
    [Iterations,value, gap, cut_gap, RC_gap] = squeeze_algo_putility(R,r,c,p,m,r_f,rav)
    CI[i] = Iterations
    FG[i] = gap
    Gap_cut[i] = cut_gap
    Gap_RC[i] = RC_gap
    [risk_df[i], obj_df[i]] = non_robust_cutting_plane(R,c,p,m,r_f,rav) - np.array([c, value])

0
rbvalue 18.150042978875984
rbvalue 17.231385953396117
RC steps 1 lowerbound -15.659701873748109
rbvalue 16.999993026012074
1
rbvalue 17.565645191150512
rbvalue 17.03550447215642
RC steps 1 lowerbound -15.73587876455118
rbvalue 16.99999967684232
2
rbvalue 18.840288528191174
rbvalue 17.245229193335994
RC steps 1 lowerbound -15.67884498522721
rbvalue 17.00410333098373
rbvalue 16.999999861398063
3
rbvalue 18.95862916229261
rbvalue 17.922313613885716
rbvalue 17.023035198796958
RC steps 1 lowerbound -15.63340700252061
rbvalue 16.99999795135881
4
rbvalue 19.635394302698913
rbvalue 17.360577357260677
RC steps 1 lowerbound -15.64024439905471
rbvalue 17.023992658741044
rbvalue 17.00000000910426
rbvalue 16.999999853182405
5
rbvalue 18.369034982021432
rbvalue 17.741327541240068
rbvalue 17.060698448530278
RC steps 1 lowerbound -15.627289184647637
rbvalue 16.999999861955715
6
rbvalue 17.198510415333203
RC steps 1 lowerbound -15.777427487364227
rbvalue 16.99999966495589
7
rbvalue 18.866322547862897

In [9]:
d = {'Cut_Iterations':CI, 'Final Gap':FG, 'Cut_Iterations Gap <= 1e-5':Gap_cut, 'RC_Iterations Gap <= 1e-5':Gap_RC, 'Robust_obj- NonRobust_obj':obj_df, 'NRB_risk - c': risk_df}
df = pd.DataFrame(data=d)

In [10]:
df

,Cut_Iterations,Final Gap,Cut_Iterations Gap <= 1e-5,RC_Iterations Gap <= 1e-5,Robust_obj- NonRobust_obj,NRB_risk - c
0,2.0,-1.428457e-08,2.0,1.0,0.004591,0.330852
1,2.0,6.111696e-08,2.0,1.0,0.002641,0.353109
2,3.0,6.024123e-10,3.0,1.0,0.014240,0.850457
3,3.0,1.048209e-08,3.0,1.0,0.011749,0.562567
4,4.0,1.128948e-08,4.0,1.0,0.029729,0.926045
5,3.0,1.080280e-07,3.0,1.0,0.006153,0.353565
6,1.0,2.003113e-08,1.0,1.0,0.000323,0.198510
7,2.0,inf,2.0,0.0,0.019760,0.414549
8,3.0,2.945722e-09,3.0,1.0,0.006046,0.607366
9,3.0,1.255486e-08,3.0,1.0,0.018011,0.496536


In [14]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna()
df

,Cut_Iterations,Final Gap,Cut_Iterations Gap <= 1e-5,RC_Iterations Gap <= 1e-5,Robust_obj- NonRobust_obj,NRB_risk - c
0,2.0,-1.428457e-08,2.0,1.0,0.004591,0.330852
1,2.0,6.111696e-08,2.0,1.0,0.002641,0.353109
2,3.0,6.024123e-10,3.0,1.0,0.014240,0.850457
3,3.0,1.048209e-08,3.0,1.0,0.011749,0.562567
4,4.0,1.128948e-08,4.0,1.0,0.029729,0.926045
5,3.0,1.080280e-07,3.0,1.0,0.006153,0.353565
6,1.0,2.003113e-08,1.0,1.0,0.000323,0.198510
8,3.0,2.945722e-09,3.0,1.0,0.006046,0.607366
9,3.0,1.255486e-08,3.0,1.0,0.018011,0.496536
10,3.0,-7.261605e-10,3.0,1.0,0.003466,0.402892


In [19]:
meandata = pd.DataFrame(data = df.mean()).transpose()
meandata

,Cut_Iterations,Final Gap,Cut_Iterations Gap <= 1e-5,RC_Iterations Gap <= 1e-5,Robust_obj- NonRobust_obj,NRB_risk - c
0,2.789474,2.619975e-08,2.789474,1.0,0.011427,0.500813
